# Solving a 2D advection Problem using PINNx

In this tutorial, we will walk through the process of solving a 2D advection problem using the PINNx (Physics-Informed Neural Networks) framework. We will demonstrate how to set up the problem, define the necessary components, and train a neural network to approximate the solution. Finally, we will compare the predicted solution with the true solution and evaluate the performance.

First, we need to import the necessary libraries. These include `brainstate`, `brainunit`, `matplotlib`, `numpy`, and `pinnx`.

In [2]:
import brainstate as bst
import brainunit as u
import matplotlib.pyplot as plt
import numpy as np

import pinnx

We will be working with a 2D problem, so we set the dimension of the space variable `x` to 5.

In [3]:
dim_x = 5

Next, we define the PDE that we want to solve. The PDE we are considering is a simple advection equation, where the time derivative of the function `y` is equal to the spatial derivative of `y`.

In [4]:
def pde(neu, x):
    approx = lambda x: neu(pinnx.dict_to_array(x))
    jacobian = pinnx.grad.jacobian(approx, pinnx.array_to_dict(x, 'x', 't'))
    dy_x = jacobian['x']
    dy_t = jacobian['t']
    return dy_t + dy_x

We define the geometry of the problem as a rectangle with corners at `[0, 0]` and `[1, 1]`. We also define the initial condition and the boundary condition.


In [5]:
geom = pinnx.geometry.Rectangle([0, 0], [1, 1])

def func_ic(x, v):
    return v


def boundary(x, on_boundary):
    return on_boundary and np.isclose(x[1], 0)


ic = pinnx.icbc.DirichletBC(geom, func_ic, boundary)

We create a PDE object that encapsulates the geometry, PDE, and boundary conditions. We also specify the number of domain and boundary points to use during training.

In [6]:
pde = pinnx.data.PDE(geom, pde, ic, num_domain=200, num_boundary=200)

We define the function space using a Gaussian Random Field (GRF) with an exponential sine squared kernel.

In [7]:
func_space = pinnx.data.GRF(kernel="ExpSineSquared", length_scale=1)

We generate the data points for training by evaluating the PDE at specific points in the domain. We also specify the number of test points and the batch size for training.

In [ ]:
eval_pts = np.linspace(0, 1, num=50)[:, None]
data = pinnx.data.PDEOperatorCartesianProd(
    pde, func_space, eval_pts, 1000, function_variables=[0], num_test=100, batch_size=32
)

We define the architecture of the neural network using a DeepONet (Deep Operator Network) with two branches. The first branch processes the input function, and the second branch processes the spatial coordinates.

In [9]:
net = pinnx.nn.DeepONetCartesianProd(
    [50, 128, 128, 128],
    [dim_x, 128, 128, 128],
    "tanh",
)

We apply a periodic feature transformation to the input data. This transformation helps the network to better capture the periodic nature of the problem.


In [11]:
def periodic(x):
    x, t = x[:, :1], x[:, 1:]
    x = x * 2 * np.pi
    return u.math.concatenate(
        [u.math.cos(x),
         u.math.sin(x),
         u.math.cos(2 * x),
         u.math.sin(2 * x),
         t],
        axis=-1
    )

net.apply_feature_transform(periodic)

We compile the model using the Adam optimizer and train it for a specified number of iterations. We also plot the loss history to monitor the training process.


In [12]:
model = pinnx.Model(data, net)
model.compile(bst.optim.Adam(0.0005))
losshistory, train_state = model.train(iterations=30000)
pinnx.utils.plot_loss_history(losshistory)

Compiling model...
'compile' took 0.037557 s

Training model...



IndexError: Too many indices: 0-dimensional array indexed with 2 regular indices.

Finally, we evaluate the model by comparing the predicted solution with the true solution. We plot both solutions and calculate the L2 relative error to quantify the performance.

In [ ]:
x = np.linspace(0, 1, num=100)
t = np.linspace(0, 1, num=100)
u_true = np.sin(2 * np.pi * (x - t[:, None]))
plt.figure()
plt.imshow(u_true)
plt.colorbar()

v_branch = np.sin(2 * np.pi * eval_pts).T
xv, tv = np.meshgrid(x, t)
x_trunk = np.vstack((np.ravel(xv), np.ravel(tv))).T
u_pred = model.predict((v_branch, x_trunk))
u_pred = u_pred.reshape((100, 100))
plt.figure()
plt.imshow(u_pred)
plt.colorbar()
plt.show()
print(pinnx.metrics.l2_relative_error(u_true, u_pred))